# 14.4 Alternating between models

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

[Chapter 13](../13/13.md) described how "scripts" of AMPL commands can be set up to run as programs
that perform repetitive actions. In several examples, a script solves a series of
related `model` instances, by including a `solve` statement inside a loop. The result is a
simple kind of sensitivity analysis algorithm, programmed in AMPL's command language.

Much more powerful algorithmic procedures can be constructed by using two models.
An optimal solution for one `model` yields new `data` for the other, and the two are solved
in alternation in such a way that some termination condition must eventually be reached.
Classic methods of column and cut generation, decomposition, and Lagrangian relaxation
are based on schemes of this kind, which are described in detail in references cited at the
end of this chapter.

To use two models in this manner, a script must have some way of switching between
them. Switching can be done with previously defined AMPL features, or more clearly and
efficiently by defining separately-named problems and environments.
We illustrate these possibilities through a script for a basic form of the "roll trim" or
"cutting stock" problem, using a well-known, elementary column-generation procedure.
In the interest of brevity, we give only a sketchy description of the procedure here, while
the references at the end of this chapter provide sources for thorough descriptions. There
are several other examples of generation, decomposition, and relaxation schemes on the
AMPL web site, and we will also use a few excerpts from them later, without showing the
whole models.

In a roll trim problem, we wish to cut up long raw widths of some commodity such as
rolls of paper into a combination of smaller widths that meet given orders with as little
waste as possible. This problem can be viewed as deciding, for each raw-width roll,
where the cuts should be made so as to produce one or more of the smaller widths that
were ordered. Expressing such a problem in terms of decision variables is awkward,
however, and leads to an `integer` program that is difficult to `solve` except for very small
instances.

To derive a more manageable `model`, the so-called Gilmore-Gomory procedure
defines a cutting pattern to be any one feasible way in which a raw roll can be cut up. A
pattern thus consists of a certain number of rolls of each desired width, such that their
total width does not exceed the raw width. If (as in Exercise 2-6) the raw width is 110",
and there are demands for widths of 20", 45", 50", 55" and 75", then two rolls of 45" and
one of 20" make an acceptable pattern, as do one of 50" and one of 55" (with 5" of
waste). Given this view, the two simple linear programs in Figure 14-2 can be made to
work together to find an efficient cutting plan.

The cutting optimization `model` ([Figure 14-2a](./14_4_named_problems.ipynb#fig-14-2a)) finds the minimum number of raw
rolls that need be cut, given a collection of known cutting patterns that may be used. This
is actually a close cousin to the diet `model`, with the variables representing patterns cut
rather than food items bought, and the constraints enforcing a lower limit on cut widths
rather than nutrients provided.

The pattern generation `model` ([Figure 14-2b](./14_4_named_problems.ipynb#fig-14-2b)) seeks to identify a new pattern that can
be used in the cutting optimization, either to reduce the number of raw rolls needed, or to
determine that no such new pattern exists. The variables of this `model` are the numbers of
each desired width in the new pattern; the single constraint ensures that the total width of
the pattern does not exceed the raw width. We won't try to explain the `objective` here,
except to note that the coefficient of a variable is given by its corresponding "dual
value" or "dual price" from the linear relaxation of the cutting optimization `model`.

We can search for a good cutting plan by solving these two problems repeatedly in
alternation. First the continuous-variable relaxation of the cutting optimization problem
generates some dual prices, then the pattern generation problem uses the prices to generate
a new pattern, and then the procedure repeats with the collection of patterns extended
by one. We stop repeating when the pattern generation problem indicates that no new
pattern can lead to an improvement. We then have the best possible solution in terms of
(possibly) fractional numbers of raw rolls cut. We may make one last run of the cutting
optimization `model` with the integrality restriction restored, to get the best integral soluparam

```ampl
roll_width > 0;               # width of raw rolls
set WIDTHS;                   # set of widths to be cut
param orders {WIDTHS} > 0;    # number of each width to be cut
param nPAT integer >= 0;      # number of patterns
set PATTERNS = 1..nPAT;       # set of patterns
param nbr {WIDTHS,PATTERNS} integer >= 0;
       check {j in PATTERNS}:
       sum {i in WIDTHS} i * nbr[i,j] <= roll_width;
                            # defn of patterns: nbr[i,j] = number
                            # of rolls of width i in pattern j
var Cut {PATTERNS} integer >= 0; # rolls cut using each pattern
minimize Number:   # minimize total raw rolls cut
       sum {j in PATTERNS} Cut[j];
subject to Fill {i in WIDTHS}:
       sum {j in PATTERNS} nbr[i,j] * Cut[j] >= orders[i];
              # for each width, total rolls cut meets total orders
```

**[Figure 14-2a](./14_4_named_problems.ipynb#fig-14-2a):** Pattern-based `model` for cutting optimization problem (cut.mod).

```ampl
param price {WIDTHS} default 0.0; # prices from cutting opt
var Use {WIDTHS} integer >= 0;
                            # numbers of each width in pattern
minimize Reduced_Cost:
       1 - sum {i in WIDTHS} price[i] * Use[i];
subject to Width_Limit:
       sum {i in WIDTHS} i * Use[i] <= roll_width;
```

**[Figure 14-2b](./14_4_named_problems.ipynb#fig-14-2b):** Knapsack `model` for pattern generation problem (cut.mod, continued).

tion using the patterns generated, or we may simply round the fractional numbers of rolls
up to the next largest integers if that gives an acceptable result.

This is the Gilmore-Gomory procedure. In terms of our two AMPL models, its steps
may be described as follows:

```ampl
pick initial patterns sufficient to meet demand
repeat
       solve the (fractional) cutting optimization problem
       let price[i] equal Fill[i].dual for each pattern i
       solve the pattern generation problem
       if the optimal value is < 0 then
              add a new pattern that cuts Use[i] rolls of each width i
       else
              find a final integer solution and stop
```

An easy way to initialize is to generate one pattern for each width, containing as many
copies of that width as will fit inside the raw roll. These patterns clearly can cover any
demands, though not necessarily in an economical way.

An implementation of the Gilmore-Gomory procedure as an AMPL script is shown in
Figure 14-3. The file cut.mod contains both the cutting optimization and pattern generation
models in Figure 14-2. Since these models have no variables or constraints in common,
it would be possible to write the script with simple `solve` statements using alternating
`objective` functions:

```ampl
repeat {
       objective Number;
       solve;
       ...
       objective Reduced_Cost;
       solve;
       ...
}
```

Under this approach, however, every `solve` would send the solver all of the variables
and constraints generated by both models. Such an arrangement is inefficient and prone
to error, especially for larger and more complex iterative procedures.

We could instead ensure that only the immediately relevant variables and constraints
are sent to the solver, by using `fix` and `drop` commands to suppress the others. Then
the outline of our loop would look like this:

```ampl
repeat {
       unfix Cut; restore Fill; objective Number;
       fix Use; drop Width_Limit;
       solve;
       ...
       unfix Use; restore Width_Limit; objective Reduced_Cost;
       fix Cut; drop Fill;
       solve;
       ...
}
```

Before each `solve`, the previously fixed variables and dropped constraints must also be
brought back, by use of `unfix` and `restore`. This approach is efficient, but it remains
highly error-prone, and makes scripts difficult to read.

As an alternative, therefore, AMPL allows models to be distinguished by use of the
problem statement seen in Figure 14-3:

```ampl
problem Cutting_Opt: Cut, Number, Fill;
option relax_integrality 1;
problem Pattern_Gen: Use, Reduced_Cost, Width_Limit;
option relax_integrality 0;
```

The first statement defines a problem named Cutting_Opt that consists of the Cut
variables, the Fill constraints, and the `objective` Number. This statement also makes

```ampl
model cut.mod;
data cut.dat;
option solver cplex, solution_round 6;
option display_1col 0, display_transpose -10;
problem Cutting_Opt: Cut, Number, Fill;
option relax_integrality 1;
problem Pattern_Gen: Use, Reduced_Cost, Width_Limit;
option relax_integrality 0;
let nPAT := 0;
for {i in WIDTHS} {
       let nPAT := nPAT + 1;
       let nbr[i,nPAT] := floor (roll_width/i);
       let {i2 in WIDTHS: i2 <> i} nbr[i2,nPAT] := 0;
}
repeat {
       solve Cutting_Opt;
       let {i in WIDTHS} price[i] := Fill[i].dual;
              solve Pattern_Gen;
              if Reduced_Cost < -0.00001 then {
              let nPAT := nPAT + 1;
              let {i in WIDTHS} nbr[i,nPAT] := Use[i];
              }
              else break;
}
display nbr, Cut;
option Cutting_Opt.relax_integrality 0;
solve Cutting_Opt;
display Cut;
```

**Figure 14-3:** Gilmore-Gomory procedure for cutting-stock problem (cut.run).

Cutting_Opt the current problem; uses of the `var`, minimize, maximize, subject
to, and option statements now apply to this problem only. Thus by setting
option `relax_integrality` to 1 above, for example, we assure that the integrality
condition on the Cut variables will be relaxed whenever Cutting_Opt is current. In a
similar way, we define a problem Pattern_Gen that consists of the Use variables, the
Width_Limit constraint, and the `objective` Reduced_Cost; this in turn becomes the
current problem, and this time we set `relax_integrality` to 0 because only `integer`
solutions to this problem are meaningful.

The for loop in Figure 14-3 creates the initial cutting patterns, after which the main
repeat loop carries out the Gilmore-Gomory procedure as described previously. The
statement

```ampl
solve Cutting_Opt;
param roll_width := 110 ;
param: WIDTHS: orders :=
              20     48
              45     35
              50     24
              55     10
              75      8 ;
```

**Figure 14-4:** Data for cutting-stock `model` (cut.dat)

restores Cutting_Opt as the current problem, along with its environment, and solves
the associated linear program. Then the assignment

```ampl
let {i in WIDTHS} price[i] := Fill[i].dual;
```

transfers the optimal dual prices from Cutting_Opt to the parameters price[i] that
will be used by Pattern_Gen. All sets and parameters are global in AMPL, so they can
be referenced or changed whatever the current problem.

The second half of the main loop makes problem Pattern_Gen and its environment
current, and sends the associated `integer` program to the solver. If the resulting
`objective` is sufficiently negative, the pattern returned by the Use[i] variables is added
to the `data` that will be used by Cutting_Opt in the next pass through the loop. Otherwise
no further progress can be made, and the loop is terminated.

The script concludes with the following statements, to `solve` for the best `integer` solution
using all patterns generated:

```ampl
option Cutting_Opt.relax_integrality 0;
solve Cutting_Opt;
```

The expression Cutting_Opt.`relax_integrality` stands for the value of the
`relax_integrality` option in the Cutting_Opt environment. We discuss these
kinds of names and their uses at greater length in the next section.

As an example of how this works, Figure 14-4 shows `data` for cutting 110" raw rolls,
to meet demands of 48, 35, 24, 10 and 8 for finished rolls of widths 20, 45, 50, 55 and 75,
respectively. Figure 14-5 shows the output that occurs when Figure 14-3's script is run
with the `model` and `data` as shown in Figures 14-2 and 14-4. The best fractional solution
cuts 46.25 raw rolls in five different patterns, using 48 rolls if the fractional values are
rounded up to the next `integer`. The final `solve` using `integer` variables shows how a
collection of six of the patterns can be applied to meet demand using only 47 raw rolls.